# LSTM on IMDb (DP-SGD)

In [ ]:
@dataclass
class ImdbConfig:
    batch_size: int = 2
    lr: float = 0.5
    epochs: int = 1
    max_grad_norm: float = 1.0
    noise_multiplier: float = 1.1
    vmap_chunk_size: Optional[int] = 8
    force_compile_mps: bool = True
    device: str = (
        "mps"
        if torch.backends.mps.is_available()
        else ("cuda" if torch.cuda.is_available() else "cpu")
    )
    vocab_min_freq: int = 2
    max_seq_len: int = 256
    embed_dim: int = 128
    lstm_hidden: int = 128
    num_classes: int = 2

imdb_cfg = ImdbConfig()

In [ ]:
import re
from datasets import load_dataset
from dotenv import load_dotenv
import os

_ = load_dotenv()
hf_token = os.getenv("HF_TOKEN")

In [ ]:


class SimpleVocab:
    """A simple vocabulary class that maps tokens to indices and vice versa."""
    def __init__(self, tokens, specials):
        self.itos = list(specials) + [t for t in tokens if t not in specials]
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.default_index = self.stoi["<unk>"]

    def __len__(self):
        return len(self.itos)

    def __getitem__(self, token):
        return self.stoi.get(token, self.default_index)

    def __call__(self, tokens):
        return [self.stoi.get(t, self.default_index) for t in tokens]

In [ ]:
# TEST
sv = SimpleVocab(tokens=["the", "cat", "sat", "on", "mat"], specials=["<pad>", "<unk>"])

In [ ]:
sv.itos

NameError: name 'sv' is not defined

In [ ]:
sv.stoi

{'<pad>': 0, '<unk>': 1, 'the': 2, 'cat': 3, 'sat': 4, 'on': 5, 'mat': 6}

In [ ]:
len(sv)

7

In [ ]:
sv["cat"]

3

In [ ]:
def simple_tokenizer(text):
    return re.findall(r"[a-z0-9']+", text.lower())

print(simple_tokenizer("The cat's$mat is on the floor."))

['the', "cat's", 'mat', 'is', 'on', 'the', 'floor']


In [ ]:
ds = load_dataset("imdb", split="train", token=hf_token)

In [ ]:
def build_imdb_vocab(tokenizer, min_freq=2):
    """Builds a vocabulary from the IMDB training set using the provided tokenizer."""
    ds = load_dataset("imdb", split="train")
    
    counts = {}
    for text in ds["text"]:
        for tok in tokenizer(text):
            counts[tok] = counts.get(tok, 0) + 1

    tokens = [t for t, c in counts.items() if c >= min_freq]
    return SimpleVocab(tokens, specials=["<pad>", "<unk>"])

In [ ]:
# TEST
vocab = build_imdb_vocab(simple_tokenizer, min_freq=2)
print(f"Vocab size: {len(vocab)}")

print(vocab["the"])
print(vocab["good", "the"])

Vocab size: 51829
14
1


In [ ]:
def get_imdb_loader(cfg, vocab, tokenizer):
    """Creates a DataLoader for the IMDB dataset that tokenizes and pads the text data."""
    def collate_batch(batch):
        labels = torch.tensor(
            [1 if item["label"] == 1 else 0 for item in batch],
            dtype=torch.long,
        )

        token_ids = [
            torch.tensor(
                vocab(tokenizer(item["text"]))[: cfg.max_seq_len], dtype=torch.long
            )
            for item in batch
        ]

        padded = nn.utils.rnn.pad_sequence(
            token_ids, batch_first=True, padding_value=vocab["<pad>"]
        )
        return padded, labels

    dataset = load_dataset("imdb", split="train", token=hf_token)

    return torch.utils.data.DataLoader(
        dataset,
        batch_size=cfg.batch_size,
        shuffle=True,
        drop_last=True,
        collate_fn=collate_batch,
    )

In [ ]:
# TEST
vocab = build_imdb_vocab(simple_tokenizer, min_freq=2)
print(f"Vocab size: {len(vocab)}")

loader = get_imdb_loader(imdb_cfg, vocab, simple_tokenizer)
batch = next(iter(loader))
print(batch[0].shape)  # token ids
print(batch[1].shape)  # labels

Vocab size: 51829
torch.Size([64, 256])
torch.Size([64])


## IMDb LSTM Training

Included: non-private SGD, DP-SGD functional (no compile), DP-SGD compiled, and Opacus DP-SGD.

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, pad_idx=0):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
        lengths = (x != self.pad_idx).sum(1).clamp(min=1)
        emb = self.embedding(x)
        out, _ = self.lstm(emb)
        idx = (lengths - 1).view(-1, 1, 1).expand(-1, 1, out.size(2))
        last = out.gather(1, idx).squeeze(1)
        return self.fc(last)

class LSTMCellClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, pad_idx=0):
        super().__init__()
        self.pad_idx = pad_idx
        self.hidden_dim = hidden_dim
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.cell = nn.LSTMCell(embed_dim, hidden_dim)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
        lengths = (x != self.pad_idx).sum(1).clamp(min=1)
        emb = self.embedding(x)
        batch_size, seq_len, _ = emb.shape
        h = emb.new_zeros(batch_size, self.hidden_dim)
        c = emb.new_zeros(batch_size, self.hidden_dim)
        for t in range(seq_len):
            h_new, c_new = self.cell(emb[:, t], (h, c))
            mask = (t < lengths).unsqueeze(1)
            h = torch.where(mask, h_new, h)
            c = torch.where(mask, c_new, c)
        return self.fc(h)

class RNNClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, pad_idx=0):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
        lengths = (x != self.pad_idx).sum(1).clamp(min=1)
        emb = self.embedding(x)
        out, _ = self.lstm(emb)
        idx = (lengths - 1).view(-1, 1, 1).expand(-1, 1, out.size(2))
        last = out.gather(1, idx).squeeze(1)
        return self.fc(last)

class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, pad_idx=0):
        super().__init__()
        self.pad_idx = pad_idx
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        self.lstm = nn.GRU(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        if x.dim() == 1:
            x = x.unsqueeze(0)
        lengths = (x != self.pad_idx).sum(1).clamp(min=1)
        emb = self.embedding(x)
        out, _ = self.lstm(emb)
        idx = (lengths - 1).view(-1, 1, 1).expand(-1, 1, out.size(2))
        last = out.gather(1, idx).squeeze(1)
        return self.fc(last)

cfg_imdb = ImdbConfig()
tokenizer = simple_tokenizer
vocab = build_imdb_vocab(tokenizer, min_freq=cfg_imdb.vocab_min_freq)
pad_idx = vocab["<pad>"]

In [ ]:
def build_imdb_model(cfg, model_name="lstm"):
    if model_name == "lstm":
        return LSTMClassifier(
            len(vocab),
            cfg.embed_dim,
            cfg.lstm_hidden,
            cfg.num_classes,
            pad_idx=pad_idx,
        ).to(cfg.device)
    elif model_name == "lstm_cell":
        return LSTMCellClassifier(
            len(vocab),
            cfg.embed_dim,
            cfg.lstm_hidden,
            cfg.num_classes,
            pad_idx=pad_idx,
        ).to(cfg.device)
    elif model_name == "rnn":
        return RNNClassifier(
            len(vocab),
            cfg.embed_dim,
            cfg.lstm_hidden,
            cfg.num_classes,
            pad_idx=pad_idx,
        ).to(cfg.device)
    elif model_name == "gru":
        return GRUClassifier(
            len(vocab),
            cfg.embed_dim,
            cfg.lstm_hidden,
            cfg.num_classes,
            pad_idx=pad_idx,
        ).to(cfg.device)
    else:
        raise ValueError(f"Unknown model name: {model_name}")


def _sync_device(device):
    if device == "cuda" and torch.cuda.is_available():
        torch.cuda.synchronize()
    elif device == "mps" and hasattr(torch, "mps") and torch.backends.mps.is_available():
        torch.mps.synchronize()


In [ ]:
# Non-private
def train_one_epoch_imdb_non_private(cfg, model_name="lstm"):
    loader = get_imdb_loader(cfg, vocab, tokenizer)
    model = build_imdb_model(cfg, model_name=model_name)

    optimizer = torch.optim.SGD(model.parameters(), lr=cfg.lr)
    criterion = nn.CrossEntropyLoss(reduction="mean")

    model.train()

    _sync_device(cfg.device)

    start = time.perf_counter()

    for x, y in loader:
        x = x.to(cfg.device)
        y = y.to(cfg.device)
        optimizer.zero_grad()

        logits = model(x)
        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()
    _sync_device(cfg.device)

    return time.perf_counter() - start

In [ ]:
# TEST
model = build_imdb_model(cfg_imdb, model_name="lstm_cell")
loader = get_imdb_loader(cfg_imdb, vocab, tokenizer)

print(model)

LSTMCellClassifier(
  (embedding): Embedding(51829, 128, padding_idx=0)
  (cell): LSTMCell(128, 128)
  (fc): Linear(in_features=128, out_features=2, bias=True)
)


In [ ]:
# TEST
params, buffers, per_sample_grads = make_per_sample_grads(model, cfg_imdb.vmap_chunk_size)
print(f"Number of parameters: {len(params)}")
print(f"Parameter names: {list(params.keys())[:5]}...")
print(f"Number of buffers: {len(buffers)}")
print(f"Buffer names: {list(buffers.keys())[:5]}...")

Number of parameters: 7
Parameter names: ['embedding.weight', 'lstm.weight_ih_l0', 'lstm.weight_hh_l0', 'lstm.bias_ih_l0', 'lstm.bias_hh_l0']...
Number of buffers: 0
Buffer names: []...


In [ ]:
def train_one_epoch_imdb_dp(cfg, model_name="lstm", use_compile=False):
    loader = get_imdb_loader(cfg, vocab, tokenizer)
    model = build_imdb_model(cfg, model_name=model_name)

    params, buffers, per_sample_grads = make_per_sample_grads(model, cfg.vmap_chunk_size)

    print(f"Number of parameters: {len(params)}")
    print(f"Parameter names: {list(params.keys())[:5]}...")

    step_fn = make_compiled_step() if use_compile else dp_sgd_step

    model.train()
    _sync_device(cfg.device)
    start = time.perf_counter()

    for x, y in loader:
        x = x.to(cfg.device)
        y = y.to(cfg.device)
        params = step_fn(params, buffers, per_sample_grads, (x, y), cfg)
        
    _sync_device(cfg.device)
    return time.perf_counter() - start

In [ ]:
cfg_imdb.device = "mps"

In [ ]:
cfg_imdb

ImdbConfig(batch_size=2, lr=0.5, epochs=1, max_grad_norm=1.0, noise_multiplier=1.1, vmap_chunk_size=8, force_compile_mps=True, device='mps', vocab_min_freq=2, max_seq_len=256, embed_dim=128, lstm_hidden=128, num_classes=2)

In [ ]:
train_one_epoch_imdb_dp(cfg_imdb, model_name="lstm_cell",use_compile=False)

Number of parameters: 7
Parameter names: ['embedding.weight', 'cell.weight_ih', 'cell.weight_hh', 'cell.bias_ih', 'cell.bias_hh']...


NameError: name 'model' is not defined

In [ ]:
# Opacus DP-SGD
def train_one_epoch_imdb_opacus(cfg):
    try:
        from opacus import PrivacyEngine
    except Exception as exc:
        print(f"Opacus not available: {exc}")
        return None
    loader = get_imdb_loader(cfg, vocab, tokenizer)
    model = build_imdb_model(cfg)
    optimizer = torch.optim.SGD(model.parameters(), lr=cfg.lr)
    privacy_engine = PrivacyEngine()
    model, optimizer, private_loader = privacy_engine.make_private(
        module=model,
        optimizer=optimizer,
        data_loader=loader,
        noise_multiplier=cfg.noise_multiplier,
        max_grad_norm=cfg.max_grad_norm,
        grad_sample_mode="hooks",
    )
    criterion = nn.CrossEntropyLoss(reduction="mean")
    model.train()
    start = time.perf_counter()

    for x, y in private_loader:
        x = x.to(cfg.device)
        y = y.to(cfg.device)
        
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
    _sync_device(cfg.device)
    return time.perf_counter() - start

In [ ]:

results_imdb = {
    # "Non-private SGD": train_one_epoch_imdb_non_private(cfg_imdb),
    "DP-SGD functional (no compile)": train_one_epoch_imdb_dp(cfg_imdb, use_compile=False),
    "DP-SGD compiled": train_one_epoch_imdb_dp(cfg_imdb, use_compile=True),
}
opacus_time = train_one_epoch_imdb_opacus(cfg_imdb)
if opacus_time is not None:
    results_imdb["Opacus DP-SGD"] = opacus_time

for k, v in results_imdb.items():
    print(f"{k}: {v:.2f}s")